# Session 1, Module 08: Functions Fundamentals


This module covers:
- def syntax, return values, multiple returns
- Parameters vs arguments
- Default parameter values
- Docstrings (Google style)
- Type hints / annotations
- Return type annotations

Data Engineering Context:
Functions are building blocks of pipelines. Well-designed functions make
code reusable, testable, and maintainable.


In [26]:
from typing import Optional, Union, Any

## Basic Function Definition


In [27]:
print("=== Basic Function Definition ===")


# Simple function
def greet():
    """Print a greeting message."""
    print("Hello, Data Engineer!")


greet()  # OUTPUT: Hello, Data Engineer!


# Function with parameter
def greet_user(name):
    """Print a personalized greeting."""
    print(f"Hello, {name}!")


greet_user("Alice")  # OUTPUT: Hello, Alice!


# Function with return value
def get_table_name(schema, table):
    """Return fully qualified table name."""
    return f"{schema}.{table}"


full_name = get_table_name("staging", "customers")
print(f"Table: {full_name}")  # OUTPUT: Table: staging.customers

=== Basic Function Definition ===
Hello, Data Engineer!
Hello, Alice!
Table: staging.customers


## Return Values


In [28]:
print("\n=== Return Values ===")


# Return a single value
def calculate_success_rate(total, failed):
    """Calculate success rate as a percentage."""
    if total == 0:
        return 0.0
    return (total - failed) / total * 100


rate = calculate_success_rate(1000, 5)
print(f"Success rate: {rate:.2f}%")  # OUTPUT: Success rate: 99.50%


# Return multiple values (as tuple)
def get_pipeline_stats(records):
    """
    Calculate statistics for a list of records.

    Returns:
        tuple: (count, minimum, maximum, average)
    """
    if not records:
        return 0, None, None, None

    count = len(records)
    minimum = min(records)
    maximum = max(records)
    average = sum(records) / count
    return count, minimum, maximum, average


values = [100, 250, 50, 300, 75]
count, min_val, max_val, avg = get_pipeline_stats(values)
print(f"Count: {count}, Min: {min_val}, Max: {max_val}, Avg: {avg}")
# OUTPUT: Count: 5, Min: 50, Max: 300, Avg: 155.0


# Early return pattern
def validate_record(record):
    """Validate a record, return error message or None if valid."""
    if not record:
        return "Record is empty"

    if "id" not in record:
        return "Missing required field: id"

    if record.get("id") <= 0:
        return "ID must be positive"

    return None  # Valid record


test_records = [
    None,
    {},
    {"name": "Test"},
    {"id": -1},
    {"id": 1, "name": "Valid"},
]

for rec in test_records:
    error = validate_record(rec)
    status = error if error else "Valid ✓"
    print(f"  {rec} → {status}")


=== Return Values ===
Success rate: 99.50%
Count: 5, Min: 50, Max: 300, Avg: 155.0
  None → Record is empty
  {} → Record is empty
  {'name': 'Test'} → Missing required field: id
  {'id': -1} → ID must be positive
  {'id': 1, 'name': 'Valid'} → Valid ✓


## Parameters Vs Arguments


In [29]:
print("\n=== Parameters vs Arguments ===")


=== Parameters vs Arguments ===


Parameters: Variables in function definition
Arguments: Values passed when calling the function

In [30]:
def process_batch(records, batch_size):  # 'records' and 'batch_size' are PARAMETERS
    """Process records in batches."""
    print(f"Processing {len(records)} records in batches of {batch_size}")


data = [1, 2, 3, 4, 5]
process_batch(data, 2)  # 'data' and '2' are ARGUMENTS

# Positional arguments: Order matters
process_batch([1, 2, 3], 10)

# Keyword arguments: Named, order doesn't matter
process_batch(batch_size=10, records=[1, 2, 3])

# Mix: Positional first, then keyword
process_batch([1, 2, 3], batch_size=10)

Processing 5 records in batches of 2
Processing 3 records in batches of 10
Processing 3 records in batches of 10
Processing 3 records in batches of 10


## Default Parameter Values


In [31]:
print("\n=== Default Parameter Values ===")


def connect_database(
    host,
    port=5432,  # Default port for PostgreSQL
    database="postgres",  # Default database
    timeout=30,  # Default timeout in seconds
):
    """Create database connection string."""
    return f"postgresql://{host}:{port}/{database}?timeout={timeout}"


# Use defaults
conn1 = connect_database("localhost")
print(f"Default: {conn1}")

# Override some defaults
conn2 = connect_database("prod-server", database="warehouse")
print(f"Custom DB: {conn2}")

# Override all
conn3 = connect_database("prod-server", 5433, "analytics", 60)
print(f"All custom: {conn3}")


=== Default Parameter Values ===
Default: postgresql://localhost:5432/postgres?timeout=30
Custom DB: postgresql://prod-server:5432/warehouse?timeout=30
All custom: postgresql://prod-server:5433/analytics?timeout=60


CAUTION: Mutable default arguments
DON'T do this:

In [32]:
def bad_append(item, items=[]):  # Mutable default!
    items.append(item)
    return items


# The list is shared across calls!
print(f"\nbad_append(1): {bad_append(1)}")  # [1]
print(f"bad_append(2): {bad_append(2)}")  # [1, 2] — Unexpected!


# DO this instead:
def good_append(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items


print(f"\ngood_append(1): {good_append(1)}")  # [1]
print(f"good_append(2): {good_append(2)}")  # [2] — Correct!


bad_append(1): [1]
bad_append(2): [1, 2]

good_append(1): [1]
good_append(2): [2]


## Docstrings (Google Style)


In [33]:
print("\n=== Docstrings (Google Style) ===")


def transform_record(
    record: dict,
    mapping: dict,
    drop_nulls: bool = False,
) -> dict:
    """
    Transform a record by applying column mapping.

    Takes a source record and transforms it according to the provided
    column mapping. Optionally removes null values from the output.

    Args:
        record: Source record as a dictionary.
        mapping: Dictionary mapping source columns to target columns.
            Keys are source column names, values are target column names.
        drop_nulls: If True, exclude keys with None values from output.
            Defaults to False.

    Returns:
        Transformed record with renamed columns.

    Raises:
        ValueError: If record is None or empty.

    Examples:
        >>> transform_record({'a': 1}, {'a': 'alpha'})
        {'alpha': 1}

        >>> transform_record({'a': 1, 'b': None}, {'a': 'x', 'b': 'y'}, drop_nulls=True)
        {'x': 1}
    """
    if not record:
        raise ValueError("Record cannot be None or empty")

    result = {}
    for source_col, target_col in mapping.items():
        if source_col in record:
            value = record[source_col]
            if drop_nulls and value is None:
                continue
            result[target_col] = value

    return result


# Test the function
source = {"first_name": "Alice", "last_name": "Smith", "middle": None}
mapping = {"first_name": "fname", "last_name": "lname", "middle": "mname"}

result = transform_record(source, mapping)
print(f"Transformed: {result}")

result_clean = transform_record(source, mapping, drop_nulls=True)
print(f"Transformed (no nulls): {result_clean}")

# Access the docstring
print(f"\nFunction docstring:\n{transform_record.__doc__[:200]}...")


=== Docstrings (Google Style) ===
Transformed: {'fname': 'Alice', 'lname': 'Smith', 'mname': None}
Transformed (no nulls): {'fname': 'Alice', 'lname': 'Smith'}

Function docstring:

    Transform a record by applying column mapping.

    Takes a source record and transforms it according to the provided
    column mapping. Optionally removes null values from the output.

    Args...


## Type Hints / Annotations


In [34]:
print("\n=== Type Hints / Annotations ===")


# Basic type hints
def get_column_index(columns: list, column_name: str) -> int:
    """Return index of column, or -1 if not found."""
    try:
        return columns.index(column_name)
    except ValueError:
        return -1


# Generic types (Python 3.9+: use built-in types directly)
def process_values(values: list[int]) -> list[int]:
    """Double each value in the list."""
    return [v * 2 for v in values]


# Dict type hints
def get_config(overrides: dict[str, Any] | None = None) -> dict[str, Any]:
    """Return config with optional overrides."""
    config = {"batch_size": 100, "timeout": 30}
    if overrides:
        config.update(overrides)
    return config


# Optional type (can be the type or None)
def find_record(records: list[dict], record_id: int) -> Optional[dict]:
    """Find record by ID, return None if not found."""
    for record in records:
        if record.get("id") == record_id:
            return record
    return None


# Union type (can be one of multiple types)
def parse_value(value: Union[str, int, float]) -> float:
    """Parse value to float."""
    return float(value)


# Test type-hinted functions
columns = ["id", "name", "email"]
print(f"Index of 'name': {get_column_index(columns, 'name')}")
print(f"Index of 'phone': {get_column_index(columns, 'phone')}")

print(f"\nDoubled values: {process_values([1, 2, 3])}")
print(f"Config: {get_config({'timeout': 60})}")

records = [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]
print(f"Find ID 1: {find_record(records, 1)}")
print(f"Find ID 99: {find_record(records, 99)}")


=== Type Hints / Annotations ===
Index of 'name': 1
Index of 'phone': -1

Doubled values: [2, 4, 6]
Config: {'batch_size': 100, 'timeout': 60}
Find ID 1: {'id': 1, 'name': 'Alice'}
Find ID 99: None


## Practical: Data Pipeline Functions


In [35]:
print("\n=== Practical: Data Pipeline Functions ===")


def clean_column_name(name: str) -> str:
    """
    Clean column name for database compatibility.

    Args:
        name: Raw column name.

    Returns:
        Cleaned column name (lowercase, underscores, no special chars).
    """
    import re

    cleaned = name.lower().strip()
    cleaned = re.sub(r"[\s\-]+", "_", cleaned)
    cleaned = re.sub(r"[^a-z0-9_]", "", cleaned)
    cleaned = re.sub(r"_+", "_", cleaned)
    return cleaned.strip("_")


def create_column_mapping(
    source_columns: list[str],
    prefix: str = "",
    suffix: str = "",
) -> dict[str, str]:
    """
    Create a mapping from source columns to cleaned target columns.

    Args:
        source_columns: List of source column names.
        prefix: Optional prefix for target column names.
        suffix: Optional suffix for target column names.

    Returns:
        Dictionary mapping source names to target names.
    """
    mapping = {}
    for col in source_columns:
        target = clean_column_name(col)
        if prefix:
            target = f"{prefix}_{target}"
        if suffix:
            target = f"{target}_{suffix}"
        mapping[col] = target
    return mapping


def validate_and_transform(
    records: list[dict],
    required_fields: list[str],
    mapping: dict[str, str],
) -> tuple[list[dict], list[dict]]:
    """
    Validate records and transform valid ones.

    Args:
        records: List of source records.
        required_fields: Fields that must be present.
        mapping: Column name mapping.

    Returns:
        Tuple of (valid_records, invalid_records).
    """
    valid = []
    invalid = []

    for record in records:
        # Check required fields
        missing = [f for f in required_fields if f not in record]
        if missing:
            invalid.append({"record": record, "error": f"Missing: {missing}"})
            continue

        # Transform column names
        transformed = {mapping.get(k, k): v for k, v in record.items()}
        valid.append(transformed)

    return valid, invalid


# Test the pipeline functions
source_columns = ["Customer ID", "First Name", "E-mail Address"]
mapping = create_column_mapping(source_columns)
print(f"Column mapping: {mapping}")

raw_records = [
    {"Customer ID": 1, "First Name": "Alice", "E-mail Address": "alice@example.com"},
    {"Customer ID": 2, "First Name": "Bob"},  # Missing email
    {"First Name": "Charlie", "E-mail Address": "charlie@example.com"},  # Missing ID
]

valid, invalid = validate_and_transform(
    raw_records,
    required_fields=["Customer ID", "E-mail Address"],
    mapping=mapping,
)

print(f"\nValid records ({len(valid)}):")
for r in valid:
    print(f"  {r}")

print(f"\nInvalid records ({len(invalid)}):")
for r in invalid:
    print(f"  {r}")


=== Practical: Data Pipeline Functions ===
Column mapping: {'Customer ID': 'customer_id', 'First Name': 'first_name', 'E-mail Address': 'e_mail_address'}

Valid records (1):
  {'customer_id': 1, 'first_name': 'Alice', 'e_mail_address': 'alice@example.com'}

Invalid records (2):
  {'record': {'Customer ID': 2, 'First Name': 'Bob'}, 'error': "Missing: ['E-mail Address']"}
  {'record': {'First Name': 'Charlie', 'E-mail Address': 'charlie@example.com'}, 'error': "Missing: ['Customer ID']"}


## Summary


In [36]:
print("\n=== Summary ===")
print("""
Function Basics:
  def function_name(param1, param2):
      '''Docstring'''
      return result

Return Values:
  - Single value: return x
  - Multiple values: return a, b, c (tuple)
  - None implicitly if no return

Parameters:
  - Positional: func(a, b)
  - Keyword: func(a=1, b=2)
  - Default: def func(x, y=10):
  - AVOID mutable defaults (use None instead)

Type Hints:
  - Basic: def func(x: int) -> str:
  - Lists: list[str]
  - Dicts: dict[str, int]
  - Optional: Optional[str] or str | None
  - Union: Union[str, int] or str | int

Docstrings (Google style):
  '''
  Short description.

  Args:
      param: Description.

  Returns:
      Description of return value.

  Raises:
      ErrorType: When this happens.
  '''
""")


=== Summary ===

Function Basics:
  def function_name(param1, param2):
      '''Docstring'''
      return result

Return Values:
  - Single value: return x
  - Multiple values: return a, b, c (tuple)
  - None implicitly if no return

Parameters:
  - Positional: func(a, b)
  - Keyword: func(a=1, b=2)
  - Default: def func(x, y=10):
  - AVOID mutable defaults (use None instead)

Type Hints:
  - Basic: def func(x: int) -> str:
  - Lists: list[str]
  - Dicts: dict[str, int]
  - Optional: Optional[str] or str | None
  - Union: Union[str, int] or str | int

Docstrings (Google style):
  '''
  Short description.

  Args:
      param: Description.

  Returns:
      Description of return value.

  Raises:
      ErrorType: When this happens.
  '''

